# DWD Ingest — Eventhouse → Lakehouse (bronze + silver + COG gold)

Reads new DWD CloudEvents from the Eventhouse `_cloudevents_dispatch` landing table (populated by the `dwd-ingest` Eventstream) and, for every file-reference event (radar composite, ICON-D2 forecast file), fetches `file_url` anonymously, writes raw bytes to `Files/bronze/<channel>/...`, and—when GIS decoders are available—publishes a Cloud Optimized GeoTIFF to `Files/gold/<channel>_cog/...` and registers it in the KQL `CogCatalog` table for Fabric Maps imagery layers.

**Bound automatically by `tools/dwd/fabric/setup.ps1`:**
- `LAKEHOUSE_PATH` — abfss URI of the default Lakehouse for this notebook
- `KUSTO_URI` / `KUSTO_DATABASE` — DWD KQL database

**Triggering:**
1. *Scheduled* (default): attach this notebook to a Fabric schedule (every N minutes); each run advances a watermark stored in `Files/_state/dwd_ingest_watermark.json`.
2. *Event-driven*: add an Activator (Reflex) destination to the Eventstream, then create an Activator rule with a *Run Fabric item* action targeting this notebook. `setup.ps1 -ActivatorName <name>` wires the Activator destination.

In [ ]:
# GIS decoders (cfgrib/eccodes/h5py/rasterio/rio-cogeo) are optional.
# When absent, the notebook still lands bronze bytes and writes a
# CogCatalog row pointing at the bronze file (no COG render).
# Install them via a Fabric Environment item attached to this notebook
# for production use — pip install of eccodes requires system binaries
# that aren't available in the default runtime.


In [ ]:
# === PARAMETERS ===
# Patched per workspace by tools/dwd/fabric/setup.ps1.

# OneLake Lakehouse root for bronze/gold files (abfss://...).
LAKEHOUSE_PATH       = "abfss://a26c1440-1c4a-4774-b944-fd62f7380d62@onelake.dfs.fabric.microsoft.com/6d214faa-c5bb-4e19-9651-d08b8bf4e628"

# Eventhouse / KQL database that holds the CogCatalog table.
KUSTO_URI            = ""
KUSTO_DATABASE       = "dwd"

# DWD upstream guidance: identify yourself in the User-Agent.
USER_AGENT           = "fabric-dwd-ingest/1.0 (+https://github.com/clemensv/real-time-sources)"

# Streaming knobs.
TRIGGER_SECONDS      = 30
FETCH_CONCURRENCY    = 16

# ICON-D2 parameters that get rendered to COG in gold (the bronze copy is
# always written regardless). Keep this list small — full archive lives in
# bronze.
ICON_D2_COG_PARAMS   = {"t_2m", "tot_prec", "td_2m", "u_10m", "v_10m", "ww"}

# Reprojection target for all gold COGs. EPSG:3857 is the native CRS of
# Azure Maps basemaps used by Fabric Maps.
COG_TARGET_CRS       = "EPSG:3857"


In [ ]:
import os, io, bz2, hashlib, json, time, datetime as _dt, traceback
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, IntegerType, TimestampType,
    DoubleType, ArrayType
)

# Optional decoders — gated so the notebook still runs on a vanilla pool.
try:
    import cfgrib  # noqa: F401
    import xarray as xr
    HAVE_CFGRIB = True
except Exception:
    HAVE_CFGRIB = False
try:
    import h5py  # noqa: F401
    HAVE_H5 = True
except Exception:
    HAVE_H5 = False
try:
    import rasterio
    from rasterio.io import MemoryFile
    from rasterio.warp import calculate_default_transform, reproject, Resampling
    from rio_cogeo.cogeo import cog_translate
    from rio_cogeo.profiles import cog_profiles
    HAVE_COG = True
except Exception:
    HAVE_COG = False

print(f"HAVE_CFGRIB={HAVE_CFGRIB}  HAVE_H5={HAVE_H5}  HAVE_COG={HAVE_COG}")


## Helpers — fetch, bronze write, COG render, KQL index update

In [ ]:
def _bronze_path(channel: str, file_name: str, *parts: str) -> str:
    return f"{LAKEHOUSE_PATH}/Files/bronze/{channel}/" + "/".join([*parts, file_name])

def _gold_path(channel: str, file_name: str, *parts: str) -> str:
    return f"{LAKEHOUSE_PATH}/Files/gold/{channel}_cog/" + "/".join([*parts, file_name])

def _onelake_url(abfss: str) -> str:
    # abfss://{ws}@onelake.dfs.fabric.microsoft.com/{lh}/path -> https://onelake.dfs.../{ws}/{lh}/path
    assert abfss.startswith('abfss://'), abfss
    rest = abfss[len('abfss://'):]
    ws, tail = rest.split('@', 1)
    host_path = tail.split('/', 1)
    host = host_path[0]
    path = host_path[1] if len(host_path) > 1 else ''
    return f"https://{host}/{ws}/{path}"

def _storage_token() -> str:
    from notebookutils import mssparkutils  # type: ignore
    try:
        return mssparkutils.credentials.getToken('storage')
    except Exception:
        return mssparkutils.credentials.getToken('https://storage.azure.com/.default')

def _write_atomic(target: str, body: bytes) -> int:
    # OneLake ADLS Gen2 REST: create file (PUT ?resource=file), append (PATCH ?action=append), flush (PATCH ?action=flush).
    url = _onelake_url(target)
    tok = _storage_token()
    h = {'Authorization': f'Bearer {tok}'}
    requests.put(url + '?resource=file', headers=h, timeout=60).raise_for_status()
    # Append in chunks of <= 4 MiB.
    chunk = 4 * 1024 * 1024
    pos = 0
    while pos < len(body):
        end = min(pos + chunk, len(body))
        requests.patch(url + f'?action=append&position={pos}', headers={**h, 'Content-Type': 'application/octet-stream'}, data=body[pos:end], timeout=120).raise_for_status()
        pos = end
    requests.patch(url + f'?action=flush&position={len(body)}', headers=h, timeout=60).raise_for_status()
    return len(body)

def _exists(path: str) -> bool:
    url = _onelake_url(path)
    tok = _storage_token()
    try:
        r = requests.head(url, headers={'Authorization': f'Bearer {tok}'}, timeout=30)
        return r.status_code == 200
    except Exception:
        return False

def _http_get(url: str) -> bytes:
    r = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=60)
    r.raise_for_status()
    return r.content


In [ ]:
def _to_cog_from_array(array_2d, transform, src_crs, target_path: str) -> dict:
    """Reproject a 2D numpy array into a Cloud Optimized GeoTIFF at target_path.
    Returns {bbox: [...], crs: 'EPSG:3857', bytes: N, sha256: ...}.
    """
    assert HAVE_COG, "rasterio/rio-cogeo not available"
    import numpy as np
    height, width = array_2d.shape
    src_profile = {
        'driver': 'GTiff', 'height': height, 'width': width, 'count': 1,
        'dtype': str(array_2d.dtype), 'crs': src_crs, 'transform': transform,
        'nodata': float('nan') if np.issubdtype(array_2d.dtype, np.floating) else None,
    }
    with MemoryFile() as mem:
        with mem.open(**src_profile) as ds:
            ds.write(array_2d, 1)
        # Reproject + COG-encode into a second memfile.
        with MemoryFile() as cog_mem:
            cog_profile = cog_profiles.get('deflate')
            cog_profile.update(dict(BIGTIFF='IF_SAFER'))
            cog_translate(
                mem.open(),
                cog_mem.name,
                cog_profile,
                in_memory=True,
                dst_kwargs={'crs': COG_TARGET_CRS},
                quiet=True,
                web_optimized=True,
                overview_resampling='average',
                overview_level=4,
            )
            cog_bytes = cog_mem.read()
    written = _write_atomic(target_path, cog_bytes)
    with MemoryFile(cog_bytes) as mf, mf.open() as ds:
        b = ds.bounds
        return {
            'bbox': [b.left, b.bottom, b.right, b.top],
            'crs': str(ds.crs),
            'bytes': written,
            'sha256': hashlib.sha256(cog_bytes).hexdigest(),
        }


In [ ]:
def _kql_ingest_cog(rows: list[dict]) -> None:
    if not rows or not KUSTO_URI or not KUSTO_DATABASE:
        return
    # Use Fabric notebook identity via mssparkutils for AAD token.
    from notebookutils import mssparkutils  # type: ignore
    try:
        tok = mssparkutils.credentials.getToken('kusto')
    except Exception:
        tok = mssparkutils.credentials.getToken('https://kusto.kusto.windows.net/.default')
    body = '\n'.join(json.dumps(r, default=str) for r in rows)
    csl = f".ingest inline into table CogCatalog with (format='multijson') <|\n{body}"
    r = requests.post(f"{KUSTO_URI}/v1/rest/mgmt",
        headers={'Authorization': f'Bearer {tok}', 'Content-Type': 'application/json'},
        json={'db': KUSTO_DATABASE, 'csl': csl}, timeout=60)
    r.raise_for_status()


## Per-channel handlers

In [ ]:
def _radar_handle(ev: dict) -> dict | None:
    data = ev.get('data') or {}
    url = data.get('file_url')
    product = data.get('product') or 'unknown'
    file_name = data.get('file_name') or os.path.basename(urlparse(url).path)
    if not url:
        return None
    modified = data.get('modified') or ev.get('time')
    dt = _dt.datetime.fromisoformat(modified.replace('Z','+00:00')) if modified else _dt.datetime.utcnow()
    parts = (f"yyyy={dt.year:04d}", f"mm={dt.month:02d}", f"dd={dt.day:02d}", f"HH={dt.hour:02d}")
    bronze = _bronze_path('radar', file_name, product, *parts)
    if not _exists(bronze):
        body = _http_get(url)
        if file_name.endswith('.bz2'):
            body = bz2.decompress(body)
            file_name = file_name[:-4]
            bronze = _bronze_path('radar', file_name, product, *parts)
        _write_atomic(bronze, body)
    # COG rendering only for ODIM-H5 sources we can decode.
    if not (HAVE_H5 and HAVE_COG and file_name.endswith('.h5')):
        return {
            'channel': 'radar', 'product': product,
            'valid_time': dt.isoformat(), 'run': '', 'lead_hour': 0,
            'parameter': '', 'level_type': '', 'level': '',
            'file_url': url, 'cog_path': bronze,
            'bbox': [], 'crs': '',
            'bytes': 0, 'sha256': '',
            'written_at': _dt.datetime.utcnow().isoformat(),
        }
    try:
        import numpy as np
        import h5py
        from rasterio.transform import from_origin
        with h5py.File(io.BytesIO(_http_get(url)), 'r') as f:
            ds = f['/dataset1/data1/data'][...]
            where = f['/where'].attrs
            ll_lon = float(where.get('LL_lon', 5.0))
            ll_lat = float(where.get('LL_lat', 47.0))
            ur_lon = float(where.get('UR_lon', 16.0))
            ur_lat = float(where.get('UR_lat', 55.0))
        h, w = ds.shape
        transform = from_origin(ll_lon, ur_lat, (ur_lon-ll_lon)/w, (ur_lat-ll_lat)/h)
        gold = _gold_path('radar', file_name.replace('.h5','.tif'), product, *parts)
        info = _to_cog_from_array(ds.astype('float32'), transform, 'EPSG:4326', gold)
    except Exception as ex:
        print(f"radar COG failed for {url}: {ex}")
        return None
    return {
        'channel': 'radar', 'product': product,
        'valid_time': dt.isoformat(), 'run': None, 'lead_hour': None,
        'parameter': None, 'level_type': None, 'level': None,
        'file_url': url, 'cog_path': gold,
        'bbox': info['bbox'], 'crs': info['crs'],
        'bytes': info['bytes'], 'sha256': info['sha256'],
        'written_at': _dt.datetime.utcnow().isoformat(),
    }


In [ ]:
def _icon_d2_handle(ev: dict) -> dict | None:
    data = ev.get('data') or {}
    url = data.get('file_url')
    if not url:
        return None
    parameter = data.get('parameter') or 'unknown'
    level_type = data.get('level_type') or 'unknown'
    level = str(data.get('level') or 'na')
    run_iso = data.get('run')
    lead = int(data.get('forecast_hour') or 0)
    file_name = data.get('file_name') or os.path.basename(urlparse(url).path)
    run_dt = _dt.datetime.fromisoformat(run_iso.replace('Z','+00:00')) if run_iso else _dt.datetime.utcnow()
    run_tag = run_dt.strftime('%Y%m%d%H')
    parts = (f"run={run_tag}", f"lead={lead:03d}", parameter, level_type, level)
    bronze = _bronze_path('icon-d2', file_name, *parts)
    payload_grib2 = None
    if not _exists(bronze):
        body = _http_get(url)
        _write_atomic(bronze, body)
        payload_grib2 = bz2.decompress(body) if file_name.endswith('.bz2') else body
    # Render to COG only for the configured parameter set and only if cfgrib is around.
    if parameter not in ICON_D2_COG_PARAMS or not (HAVE_CFGRIB and HAVE_COG):
        return {
            'channel': 'icon-d2', 'product': '',
            'valid_time': (run_dt + _dt.timedelta(hours=lead)).isoformat(),
            'run': run_dt.isoformat(), 'lead_hour': lead,
            'parameter': parameter, 'level_type': level_type, 'level': level,
            'file_url': url, 'cog_path': bronze,
            'bbox': [], 'crs': '',
            'bytes': 0, 'sha256': '',
            'written_at': _dt.datetime.utcnow().isoformat(),
        }
    try:
        if payload_grib2 is None:
            body = _http_get(url)
            payload_grib2 = bz2.decompress(body) if file_name.endswith('.bz2') else body
        # cfgrib needs a real path.
        scratch = f"/tmp/{run_tag}_{lead:03d}_{parameter}_{level}.grib2"
        with open(scratch, 'wb') as f:
            f.write(payload_grib2)
        ds = xr.open_dataset(scratch, engine='cfgrib')
        var = list(ds.data_vars)[0]
        arr = ds[var].values
        if arr.ndim == 3:  # collapse leading singletons
            arr = arr[0]
        lats = ds['latitude'].values
        lons = ds['longitude'].values
        from rasterio.transform import from_bounds
        transform = from_bounds(float(lons.min()), float(lats.min()),
                                float(lons.max()), float(lats.max()),
                                arr.shape[1], arr.shape[0])
        gold = _gold_path('icon-d2', file_name.replace('.grib2.bz2', '.tif').replace('.grib2', '.tif'), *parts)
        info = _to_cog_from_array(arr.astype('float32'), transform, 'EPSG:4326', gold)
    except Exception as ex:
        print(f"icon-d2 COG failed for {url}: {ex}")
        return None
    return {
        'channel': 'icon-d2',
        'product': f"{parameter}/{level_type}/{level}",
        'valid_time': (run_dt + _dt.timedelta(hours=lead)).isoformat(),
        'run': run_dt.isoformat(), 'lead_hour': lead,
        'parameter': parameter, 'level_type': level_type, 'level': level,
        'file_url': url, 'cog_path': gold,
        'bbox': info['bbox'], 'crs': info['crs'],
        'bytes': info['bytes'], 'sha256': info['sha256'],
        'written_at': _dt.datetime.utcnow().isoformat(),
    }


## Driver: watermark-driven KQL pull + `process_batch`

Each notebook run reads new rows from `_cloudevents_dispatch` since the stored watermark, classifies each event by CloudEvents `type`, dispatches to the per-channel handler with a bounded thread pool, writes any resulting `CogCatalog` rows in one KQL ingest, and advances the watermark.

In [ ]:
def _classify(row) -> str:
    t = (row.get('type') or '').lower()
    if t.endswith('.radarfileproduct'):     return 'radar'
    if t.endswith('.icond2forecastfile'):   return 'icon-d2'
    return 'ignore'

def _dispatch(row: dict):
    kind = _classify(row)
    if kind == 'radar':   return _radar_handle(row)
    if kind == 'icon-d2': return _icon_d2_handle(row)
    return None

def process_batch(rows: list, epoch_id):
    if not rows:
        return {'events': 0, 'cog_rows': 0, 'errors': []}
    cog_rows: list[dict] = []
    errors: list[str] = []
    with ThreadPoolExecutor(max_workers=FETCH_CONCURRENCY) as pool:
        for fut in as_completed(pool.submit(_dispatch, r) for r in rows):
            try:
                out = fut.result()
                if out is not None:
                    cog_rows.append(out)
            except Exception as ex:
                errors.append(f"{type(ex).__name__}: {ex}")
                print(f"batch handler error: {ex}\n{traceback.format_exc()}")
    if cog_rows:
        try:
            _kql_ingest_cog(cog_rows)
        except Exception as ex:
            errors.append(f"cog_ingest: {type(ex).__name__}: {ex}")
    print(f"epoch={epoch_id} events={len(rows)} cog_rows={len(cog_rows)} errors={len(errors)}")
    return {'events': len(rows), 'cog_rows': len(cog_rows), 'errors': errors}


In [ ]:
# Watermark stored in the Lakehouse so each run picks up where the last left off.
import json as _json
from datetime import datetime, timezone, timedelta
from notebookutils import mssparkutils  # type: ignore

WATERMARK_PATH = f"{LAKEHOUSE_PATH}/Files/_state/dwd_ingest_watermark.json"

def _load_watermark() -> str:
    try:
        raw = mssparkutils.fs.head(WATERMARK_PATH, 4096)
        return _json.loads(raw)['since']
    except Exception:
        # First run: start from 'now - 1 hour' so the very first invocation
        # picks up recent dispatch rows without doing a full backfill.
        return ((datetime.now(timezone.utc) - timedelta(hours=1))
                .replace(microsecond=0).isoformat().replace('+00:00', 'Z'))

def _save_watermark(ts: str) -> None:
    payload = _json.dumps({'since': ts})
    mssparkutils.fs.put(WATERMARK_PATH, payload, overwrite=True)

def _kql_query(csl: str) -> list[dict]:
    # Fabric notebooks: use mssparkutils.credentials for AAD tokens.
    try:
        tok = mssparkutils.credentials.getToken('kusto')
    except Exception:
        tok = mssparkutils.credentials.getToken('https://kusto.kusto.windows.net/.default')
    resp = requests.post(f"{KUSTO_URI}/v1/rest/query",
        headers={'Authorization': f'Bearer {tok}'},
        json={'db': KUSTO_DATABASE, 'csl': csl}, timeout=120)
    resp.raise_for_status()
    tables = resp.json().get('Tables') or resp.json().get('tables') or []
    if not tables: return []
    t = tables[0]
    cols = [c['ColumnName'] if 'ColumnName' in c else c['name'] for c in t.get('Columns') or t.get('columns')]
    return [dict(zip(cols, r)) for r in (t.get('Rows') or t.get('rows') or [])]

import traceback as _tb
_DIAG_PATH = f"{LAKEHOUSE_PATH}/Files/_state/dwd_ingest_last_run.txt"
_diag = []
def _log(msg):
    line = f"{datetime.now(timezone.utc).isoformat()} {msg}"
    print(line)
    _diag.append(line)
try:
    _log('driver: start')
    since = _load_watermark()
    _log(f'reading _cloudevents_dispatch since {since}')
    csl = (f"_cloudevents_dispatch "
           f"| where ['time'] > datetime({since}) "
           f"| where type startswith 'DE.DWD.' "
           f"| project ['time'], specversion, type, source, ['id'], subject, data "
           f"| order by ['time'] asc")
    rows = _kql_query(csl)
    _log(f'kql returned {len(rows)} rows')
    stats = process_batch(rows, epoch_id=since)
    _log(f'process_batch stats: {stats}')
    if rows:
        _save_watermark(rows[-1]['time'])
        _log(f'watermark advanced to {rows[-1]["time"]}')
    _log('driver: ok')
except Exception as _e:
    _log('driver: FAILED ' + repr(_e))
    _diag.append(_tb.format_exc())
    try:
        mssparkutils.fs.put(_DIAG_PATH, '\n'.join(_diag), overwrite=True)
    except Exception:
        pass
    raise
finally:
    try:
        mssparkutils.fs.put(_DIAG_PATH, '\n'.join(_diag), overwrite=True)
    except Exception:
        pass
